# 01 – Tiền xử lý ViMD → 16 kHz mono, lưu lên Google Drive

- **Runtime:** CPU + High-RAM là đủ (không cần GPU, đỡ tốn compute unit).
- **Thời gian:** khoảng 30–60 phút. Chỉ cần chạy **một lần**.
- **Bị ngắt giữa chừng:** chỉ cần *Run all* lại, các shard đã xong sẽ tự bỏ qua.
- **Kết quả trên Drive** (`MyDrive/vimd_dl/`):
  - `data16k/*.parquet`: audio FLAC 16 kHz mono kèm metadata
  - `metadata.csv`, `provinces.csv`, `prep_report.json`

In [ ]:
# ==== CẤU HÌNH ====
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
REPO_ID = "nguyendv02/ViMD_Dataset"
RAW_SOURCE = None      # None = tải từ Hugging Face; hoặc đường dẫn thư mục parquet gốc có sẵn
WORKERS = None         # None = tự chọn theo CPU/RAM
TARGET_SR = 16000
AUDIO_COL = "audio"
PROVINCE_COL, SPEAKER_COL = "province_code", "speakerID"

In [ ]:
import os, io, glob, json, time, shutil, warnings, multiprocessing as mp
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq, soundfile as sf, librosa
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

LOCAL = "/content" if os.path.isdir("/content") else os.path.abspath("_work")
OUT_DIR, RAW_TMP, OUT_TMP = f"{DRIVE_ROOT}/data16k", f"{LOCAL}/vimd_raw_tmp", f"{LOCAL}/vimd16k_tmp"
for d in (OUT_DIR, RAW_TMP, OUT_TMP):
    os.makedirs(d, exist_ok=True)

free_gb = lambda p: shutil.disk_usage(p).free / 1e9
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
WORKERS = WORKERS or max(1, min(os.cpu_count(), int(ram_gb // 3)))
print(f"Trống: Drive {free_gb(DRIVE_ROOT):.0f} GB | local {free_gb(LOCAL):.0f} GB")
print(f"CPU {os.cpu_count()} | RAM {ram_gb:.0f} GB | workers {WORKERS}")

## 1. Danh sách shard gốc

In [ ]:
if RAW_SOURCE:
    files = sorted(os.path.relpath(f, RAW_SOURCE)
                   for f in glob.glob(f"{RAW_SOURCE}/**/*.parquet", recursive=True))
else:
    from huggingface_hub import HfApi, hf_hub_download
    files = sorted(f for f in HfApi().list_repo_files(REPO_ID, repo_type="dataset", token=token)
                   if f.endswith(".parquet"))

def split_of(path):
    parts = [p.lower() for p in path.replace("\\", "/").split("/")]
    for key, val in [("train", "train"), ("valid", "valid"), ("dev", "valid"), ("test", "test")]:
        if any(p.startswith(key) for p in parts):
            return val
    return "unknown"

def out_name(rel):
    return f"{split_of(rel)}__{os.path.basename(rel)}"

done = sum(os.path.exists(f"{OUT_DIR}/{out_name(f)}") for f in files)
print(f"{len(files)} shard gốc | đã xử lý: {done}")
print(pd.Series([split_of(f) for f in files]).value_counts().to_dict())

## 2. Chuyển đổi (song song theo shard)

In [ ]:
def decode(cell):
    """Trả về waveform float32 mono 16 kHz."""
    if cell.get("array") is not None:
        y, sr = np.asarray(cell["array"], dtype=np.float32), cell.get("sampling_rate") or TARGET_SR
    else:
        b = cell["bytes"]
        try:
            y, sr = sf.read(io.BytesIO(b), dtype="float32", always_2d=False)
        except Exception:
            import tempfile
            ext = os.path.splitext(cell.get("path") or "")[1] or ".mp3"
            with tempfile.NamedTemporaryFile(suffix=ext) as tmp:
                tmp.write(b); tmp.flush()
                y, sr = librosa.load(tmp.name, sr=None, mono=True)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR, res_type="soxr_hq")
    return np.clip(y, -1.0, 1.0).astype(np.float32)

def to_flac(y):
    buf = io.BytesIO()
    sf.write(buf, y, TARGET_SR, format="FLAC", subtype="PCM_16")
    return buf.getvalue()

def convert(rel):
    name = out_name(rel)
    dst = f"{OUT_DIR}/{name}"
    if os.path.exists(dst):
        return name, "skip", 0, 0, 0.0
    t = time.time()
    if RAW_SOURCE:
        src = os.path.join(RAW_SOURCE, rel)
    else:
        src = hf_hub_download(REPO_ID, rel, repo_type="dataset", local_dir=RAW_TMP, token=token)
    tab = pq.read_table(src)
    meta = tab.drop([AUDIO_COL]).to_pandas()
    col = tab.column(AUDIO_COL)
    keep, audio, dur, fails = [], [], [], 0
    for i in range(len(meta)):
        try:
            y = decode(col[i].as_py())
            if len(y) < TARGET_SR // 4:
                raise ValueError("audio quá ngắn")
            audio.append(to_flac(y)); dur.append(len(y) / TARGET_SR); keep.append(i)
        except Exception as e:
            fails += 1
            print(f"  lỗi {rel}#{i}: {repr(e)[:120]}", flush=True)
    meta = meta.iloc[keep].reset_index(drop=True)
    meta["split"], meta["src_file"], meta["src_row"] = split_of(rel), rel, keep
    meta["duration"] = np.asarray(dur, dtype=np.float32)
    out = pa.Table.from_pandas(meta, preserve_index=False).append_column(
        "audio", pa.array(audio, type=pa.binary()))
    local = f"{OUT_TMP}/{name}"
    pq.write_table(out, local)
    shutil.copy(local, dst + ".part"); os.replace(dst + ".part", dst)
    os.remove(local)
    if not RAW_SOURCE:
        os.remove(src)
    return name, "ok", len(keep), fails, time.time() - t

t0 = time.time(); stats = []
with mp.get_context("fork").Pool(WORKERS) as pool:
    for k, r in enumerate(pool.imap_unordered(convert, files), 1):
        stats.append(r)
        if r[1] == "ok":
            print(f"[{k}/{len(files)}] {r[0]}: {r[2]} câu, lỗi {r[3]}, {r[4]:.0f}s "
                  f"| tổng {(time.time() - t0) / 60:.1f} phút", flush=True)
print("Xong sau %.1f phút" % ((time.time() - t0) / 60))

## 3. Metadata, bảng tỉnh và kiểm tra trùng người nói

In [ ]:
outs = sorted(glob.glob(f"{OUT_DIR}/*.parquet"))
assert len(outs) == len(files), f"Mới có {len(outs)}/{len(files)} shard – chạy lại ô trên."
cols = [c for c in pq.read_schema(outs[0]).names if c != "audio"]
meta = pd.concat([pq.read_table(f, columns=cols).to_pandas().assign(
    shard=os.path.basename(f), row=lambda d: np.arange(len(d))) for f in outs], ignore_index=True)

meta["spk"] = meta[PROVINCE_COL].astype(str) + "|" + meta[SPEAKER_COL].astype(str)
sets = {s: set(g["spk"]) for s, g in meta.groupby("split")}
overlap = {f"{a}&{b}": sorted(sets[a] & sets[b]) for a in sets for b in sets if a < b}
# loại khỏi valid những người nói cũng xuất hiện trong test/train
leak_spk = (sets.get("valid", set()) & (sets.get("test", set()) | sets.get("train", set())))
meta["exclude"] = (meta["split"] == "valid") & meta["spk"].isin(leak_spk)
meta.to_csv(f"{DRIVE_ROOT}/metadata.csv", index=False)

prov = meta.groupby(PROVINCE_COL).agg(
    province_name=("province_name", lambda s: s.mode().iloc[0]),
    region=("region", lambda s: s.mode().iloc[0]),
    n_utt=("spk", "size"), n_spk=("spk", "nunique"),
    hours=("duration", lambda s: round(float(s.sum()) / 3600, 2))).reset_index()
prov.to_csv(f"{DRIVE_ROOT}/provinces.csv", index=False)

prep = dict(
    n_utt=len(meta), n_fail=int(sum(r[3] for r in stats)), n_provinces=int(meta[PROVINCE_COL].nunique()),
    hours_per_split=(meta.groupby("split")["duration"].sum().astype(float) / 3600).round(2).to_dict(),
    utt_per_split=meta["split"].value_counts().to_dict(),
    duration_sec=meta["duration"].describe(percentiles=[.05, .5, .95]).round(2).to_dict(),
    speaker_overlap={k: len(v) for k, v in overlap.items()}, n_valid_excluded=int(meta["exclude"].sum()),
    data16k_gb=round(sum(os.path.getsize(f) for f in outs) / 1e9, 2),
    regions=meta["region"].value_counts().to_dict(),
)
json.dump(prep, open(f"{DRIVE_ROOT}/prep_report.json", "w"), ensure_ascii=False, indent=1, default=str)
print(json.dumps(prep, ensure_ascii=False, indent=1, default=str))
print(prov.head(10).to_string())